In [7]:
from rich.pretty import pprint

## data preparation

```
[{'content': 'You are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.',
  'role': 'system'},
 {'content': "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Let's think step by step and output the final answer after `####`.",
  'role': 'user'}]
```

### rlhf-dataset

In [6]:
# tokenizer.apply_chat_template(doc[prompt_key], add_generation_prompt=True)
prompts = "<|im_start|>system\nYou are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.<|im_end|>\n<|im_start|>user\nNatalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Let's think step by step and output the final answer after `####`.<|im_end|>\n<|im_start|>assistant\n"
print(prompts)

<|im_start|>system
You are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.<|im_end|>
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? Let's think step by step and output the final answer after `####`.<|im_end|>
<|im_start|>assistant



## tool

```sh
PROJECT_DIR="$(pwd)"
CONFIG_PATH="$PROJECT_DIR/examples/sglang_multiturn/config"


--config-path="$CONFIG_PATH" \
--config-name='gsm8k_multiturn_grpo' \
actor_rollout_ref.rollout.multi_turn.tool_config_path="$PROJECT_DIR/examples/sglang_multiturn/config/tool_config/gsm8k_tool_config.yaml" \
actor_rollout_ref.rollout.multi_turn.interaction_config_path="$PROJECT_DIR/examples/sglang_multiturn/config/interaction_config/gsm8k_interaction_config.yaml" \
actor_rollout_ref.rollout.multi_turn.max_user_turns=1
```

- `actor_rollout_ref.rollout.multi_turn.tool_config_path`
    - examples/sglang_multiturn/config/tool_config/gsm8k_tool_config.yaml
- `gsm8k_multiturn_grpo.yaml`
    ```
    actor_rollout_ref:
      hybrid_engine: True
      rollout:
        name: sglang
        multi_turn:
          enable: True
          max_assistant_turns: 5
    ```    
- verl/tools/gsm8k_tool.py
    ```python
    class Gsm8kTool(BaseTool):
        async def create
        async def execute
    ```
- verl/interactions/gsm8k_interactions.py


In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/home/zhangchunhui/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/.no_exist'
Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/home/zhangchunhui/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/.no_exist'
Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/home/zhangchunhui/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/.no_exist'


In [10]:
messages = [{'role': 'system', 
             'content': 'You are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.'},
            {'role': 'user', 'content': "Lee rears only sheep and geese on his farm.  If the total number of animal legs is 70, and the total number of animal heads is 20, how many sheep live on Lee's farm? Let's think step by step and output the final answer after `####`."}
           ]

In [15]:
tool_schema = {'type': 'function',
 'function': {'name': 'calc_gsm8k_reward',
  'description': 'A tool for calculating the reward of gsm8k. (1.0 if parsed answer is correct, 0.0 if parsed answer is incorrect or not correctly parsed)',
  'parameters': {'type': 'object',
   'properties': {'answer': {'type': 'string',
     'description': "The model's answer to the GSM8K math problem, must be a digits",
     'enum': None}},
   'required': ['answer']},
  'strict': False}}

In [16]:
print(tokenizer.apply_chat_template(messages, tools=[tool_schema], add_generation_prompt=False, tokenize=False))

<|im_start|>system
You are a math expert. You are given a question and you need to solve it step by step. Reasoning step by step before any tool call. You should use the `calc_gsm8k_reward` tool after step by step solving the question, before generate final answer at least once and refine your answer if necessary. Put your final answer in the format of `#### <answer>`.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "calc_gsm8k_reward", "description": "A tool for calculating the reward of gsm8k. (1.0 if parsed answer is correct, 0.0 if parsed answer is incorrect or not correctly parsed)", "parameters": {"type": "object", "properties": {"answer": {"type": "string", "description": "The model's answer to the GSM8K math problem, must be a digits", "enum": null}}, "required": ["answer"]}, "strict": false}}
</tools>

For each function call,

## multi-turn rollout

- SGLangRollout: 负责高层级的流程编排（状态切换、循环、调用引擎）。
    - generate_sequences
        - _req_level_generate_sequences: 为所有的 request 构建 AsyncRolloutRequest 对象
            - AsyncRolloutRequest:
                - `state=AsyncRolloutRequestStateEnum.PENDING`,
        - 通过 asyncio 为每个 _req 调用 _async_rollout_a_request
        - （_req_level_generate_sequences）的输出
            - prompts: 只包含最开始的用户问题的 tokenized 表示。
            - responses: 从 prompt 结束之后发生的所有事情，按顺序拼接而成。
                - `[ <Assistant_Turn_1_tokens> | <Tool_Response_tokens> | <Assistant_Turn_2_tokens> | <padding> ]`
            - response_mask
                - `[ <1, 1, 1, ...> | <0, 0, 0, ...> | <1, 1, 1, ...> | <0, 0, 0, ...> ]`
```python
req_list = self._preprocess_prompt_to_async_rollout_requests(
    prompts,
)
loop = asyncio.get_event_loop()
output_req_list = loop.run_until_complete(
    asyncio.gather(
        *[self._async_rollout_a_request(req, do_sample, is_validate, **kwargs) for req in req_list],
    )
)
sorted_output_req_list = sorted(output_req_list, key=lambda x: (x.batch_data_id, x.rollout_offset))
```

```python
# <|im_start|> 等 special token 都被 skip 了
prompt_str = self.tokenizer.decode(valid_prompt_ids, skip_special_tokens=True)
response_str = self.tokenizer.decode(valid_response_ids, skip_special_tokens=True)
```

- 一个具体的 response str 如下：

```
To solve this problem, let's break it down step by step:

1. We start with 36 penguins sunbathing in the snow.

2. One-third of them jump in and swim in the ocean. 
   - One-third of 36 is \( \frac{1}{3} \times 36 = 12 \).
   - So, 12 penguins jump in and swim in the ocean.

3. Another one-third go inside the cave to eat their dinner.
   - Another one-third of 36 is \( \frac{1}{3} \times 36 = 12 \).
   - So, 12 penguins go inside the cave to eat their dinner.

4. Now, let's calculate how many penguins are left sunbathing.
   - We started with 36 penguins.
   - 12 penguins jumped in and swam in the ocean.
   - 12 penguins went inside the cave to eat their dinner.
   - Therefore, the number of penguins still left sunbathing is \( 36 - 12 - 12 = 12 \).

5. We need to check if the answer is correct by calling the `calc_gsm8k_reward` function with the answer "12".
<tool_call>
{"name": "calc_gsm8k_reward", "arguments": {"answer": "12"}}
</tool_call>
user
<tool_response>
Current parsed answer='12' reward=1.0
</tool_response>
assistant
The `calc_gsm8k_reward` function has been called with the answer "12" and the reward is 1.0, indicating that the answer is correct.

Therefore, the final answer is 12.
user
Your response is incorrect! You need to reflect on your answer and try again.
assistant
I apologize for the mistake. Let's re-evaluate the problem step by step:

1. We start with 36 penguins sunbathing in the snow.

2. One-third of them jump in and swim in the ocean.
   - One-third of 36 is \( \frac{1}{3} \times 36 = 12 \).
   - So, 12 penguins jump in and swim in the ocean.

3. Another one-third go inside the cave to eat their dinner.
   - Another one-third of 36 is \( \frac{1}{3} \times 36 = 12 \).
   - So, 12 penguins go inside the cave to eat their dinner.

4. Now, let's calculate how many penguins are left sunbathing.
   - We started with 36 penguins.
   - 12 penguins jumped in and swam in the ocean.
   - 12 penguins went inside the cave to eat their dinner.
   - Therefore, the number of penguins still left sunbathing is \( 36 - 12 - 12 = 12 \).

5. We need to check if the answer is correct by calling the `calc_gsm8k_reward` function with the answer "12".
<tool_call>
{"name": "calc_gsm8k_reward", "arguments": {"answer": "12"}}
</tool_call>
user
<tool_response>
Current parsed answer='12' reward=1.0
</tool_response>
assistant
The `calc_gsm8k_reward` function has been called with the answer "12" and the reward is 1.0, indicating that the answer is correct.

Therefore, the final answer is 12.
```

### _async_rollout_a_request

- 依然是异步的 `async def _async_rollout_a_request`

```mermaid
graph TD;
    subgraph "Conversation Lifecycle in _async_rollout_a_request"
        Start --> PENDING;

        PENDING -- "Initialize Tools & Interactions" --> RUNNING;

        RUNNING -- "Generate with LLM" --> Decision{"Analyze LLM Output"};

        Decision -- "Normal Text or Stop Token" --> COMPLETED;
        Decision -- "Tool Call Generated" --> TOOL_CALLING;
        Decision -- "Interaction Needed" --> INTERACTING;

        TOOL_CALLING -- "Execute Tool, Add Result to History" --> RUNNING;
        INTERACTING -- "Simulate Response, Add to History" --> RUNNING;
        
        COMPLETED["COMPLETED<br/>(Loop Ends)"];
        COMPLETED --> Finalize["Calculate Rewards & Release Resources"];
        Finalize --> End;
    end
```

## reward

## 训练脚本

```
unset ROCR_VISIBLE_DEVICES
unset HIP_VISIBLE_DEVICES
```
- `(TaskRunner pid=301806) TypeError: ServerArgs.__init__() got an unexpected keyword argument 'mm_attention_backend'`
    - `pip install sglang[all]==0.4.6.post3`

- `actor_rollout_ref.rollout.name=sglang`

### 增加一些调试